# Multigrain Tutorial

This tutorial generates a tetrahedral four-grain geometry with an explicit grain-boundary phase, explores the mesh interactively with PyVista, and converts it into a region-aware Ubermag system for TOMMOS.

In [ ]:
from contextlib import chdir
from pathlib import Path

import matplotlib.pyplot as plt
import micromagneticmodel as mm
import numpy as np
import pyvista as pv

import tommos_adapter as ta

# Trame provides the interactive PyVista view inside Jupyter.
pv.set_jupyter_backend("trame")

## Generate a multigrain mesh

TOMMOS delegates the polycrystal construction to Neper and then creates a tetrahedral volume mesh. The command below generates a $40\,\mathrm{nm}$ cube containing four grains. `--h` controls the target grain-mesh spacing, while `--gb-thickness` and `--gb-h` control the explicit grain-boundary phase. `--id 7` selects a fixed Neper seed for the microstructure.

Two files are written: `multigrain.npz` contains the TOMMOS solver mesh and `multigrain.vtu` contains the equivalent mesh for visualization and further processing with PyVista.

In [ ]:
!tommos mesh \
    --geom poly_gb \
    --extent 40,40,40 \
    --h 3 \
    --n 4 \
    --id 7 \
    --gb-thickness 2 \
    --gb-h 1.5 \
    --out-name multigrain

## Inspect the generated mesh using PyVista

### Import the VTU file and check its regions

The VTU file stores the tetrahedral mesh and one material ID for each cell. The following summary reports the number of points and cells, the spatial extents, and the number of tetrahedra belonging to every material. This is a useful first consistency check before assigning physical parameters.

In [ ]:
mesh = pv.read("multigrain.vtu")
# TOMMOS stores one material identifier per tetrahedron.
material_ids = np.asarray(mesh.cell_data["mat_id"])

mesh

In [ ]:
print("Material IDs:", np.unique(material_ids))

ids, counts = np.unique(material_ids, return_counts=True)
for material_id, count in zip(ids, counts, strict=True):
    print(f"Material {material_id}: {count} tetrahedra")

### Explore two-dimensional slices of the tetrahedral mesh

A full tetrahedral mesh can be difficult to inspect from the outside. The following plot therefore shows an $xy$ section through the mesh. Move the slider to select the section's $z$ coordinate; colors denote the material IDs and the black box marks the full mesh extent.

In [ ]:
# TODO: first show the full 3d grain structure and then second the sliced version
# TODO: why does the error message appear in this plot? (python 3.11??, switch to python3.14!! -> create independent issue for this change)
pl = pv.Plotter()
pl.add_mesh(mesh.outline(), color="black", line_width=2)
pl.add_mesh_clip_plane(mesh, normal="z", invert=True, scalars="mat_id")
pl.show()


### Inspect the grains and grain boundary in three dimensions

The next view separates the magnetic grains from the grain-boundary phase. The grains are colored by material ID, while the grain boundary is rendered transparently so that the internal microstructure remains visible.

In [ ]:
# The poly_gb mesher assigns the final material ID to the grain boundary.
mesh.threshold(4, method="lower").plot()

## Setting the initial magnetization

The generated coordinates are expressed in nanometres, whereas Ubermag models use SI units. The adapter also expects the cell-wise material identifiers under `region_id`, optional human-readable region metadata in `field_data`, and a unit magnetization vector `m` at every mesh point.

In [ ]:
ids = np.unique(mesh["mat_id"])
grain_boundary_id = ids.max()
names = [f"grain_{i}" if i != grain_boundary_id else "grain_boundary" for i in ids]

# Region names allow energy parameters to be assigned with dictionaries.
mesh.field_data["mat_ids"] = ids
mesh.field_data["mat_names"] = names

# TODO: decide on which option is better 
mesh.field_data["mesh_unit_length"] = 1e-9  # in meters
mesh.field_data["mesh_scale"] = 1e-9  # in meters

# Start from a uniform, normalized magnetization along +z.
mesh.point_data["m"] = np.tile((0, 0, 1), (mesh.n_points, 1))

## Define the micromagnetic system using Ubermag

The material parameters below are illustrative. The four grains are magnetic and share the same material parameters and easy-axis direction. The grain-boundary phase is non-magnetic. Region dictionaries connect the named PyVista regions to the corresponding Ubermag energy parameters.

In [ ]:
Ms = {"default": 1e6, "grain_boundary": 0.0}
A = {"default": 1e-11, "grain_boundary": 0.0}
K = {"default": 4e5, "grain_boundary": 0.0}
u = {
    "grain_1": (0, 0, 1),
    "grain_2": (0, 1, 0),
    "grain_3": (1, 0, 0),
    "grain_4": (0, 0, 1),
#    "grain_boundary": (0, 0, 1),  # TODO: ?? grain boundary needs a direction different from zero-vector
}

mesh.cell_data["Ms"] = np.full(mesh.n_points, Ms["default"])
mesh.cell_data["Ms"][mesh.mat_id==grain_boundary_id] = Ms["grain_boundary"]

system = mm.System(name="multigrain")
system.m = mesh
system.energy = mm.Exchange(A=A) + mm.UniaxialAnisotropy(K=K, u=u) + mm.Demag()
system

## Prepare a hysteresis calculation

The following field sweep starts at $+2\,\mathrm{T}$ along the easy axis and ends at $-2\,\mathrm{T}$. Five field points and relaxed minimizer settings are sufficient for an initial technical smoke test; they are not intended to produce a well-resolved physical hysteresis curve.

In [ ]:
mu0 = mm.consts.mu0
hd = ta.HysteresisDriver()
hd.drive(system, h_direction=(0, 0, 1), h_start=2.0/mu0, h_final=-2.0/mu0, n=30)

In [ ]:
system.m

### Plot the magnetization vector field without the Airbox

For the vector-field view, only cells belonging to the magnetic grains are retained. This removes both the Airbox and the non-magnetic grain-boundary phase. PyVista's `glyph` filter converts the nodal `m` vectors into arrows. `tolerance` reduces the arrow density to keep the interactive plot responsive, and the arrow color represents the $z$ component $m_z$.

In [ ]:
# NBVAL_SKIP
# TODO: magnetization should be directly called form system.m, not magnetic_result_nm

magnetic_result_nm.point_data["m_z"] = np.asarray(
    magnetic_result_nm.point_data["m"]
)[:, 2]

magnetization_glyphs = magnetic_result_nm.glyph(
    orient="m",
    scale=False,
    factor=2.0,
    tolerance=0.04,
)

# TODO: hide colorbar for the grains
vector_plotter = pv.Plotter()
vector_plotter.add_mesh(
    magnetic_result_nm.extract_surface(algorithm="dataset_surface"),
    scalars="mat_id",
    categories=True,
    opacity=0.15,
)
# TODO: pass mz directly instead of selection above
vector_plotter.add_mesh(
    magnetization_glyphs,
    scalars="m_z",
    cmap="coolwarm",
    clim=(-1, 1),
    scalar_bar_args={"title": "m_z"},
)
vector_plotter.add_axes()
vector_plotter.show()

### Plot the average magnetization of each grain

Instead of displaying the complete nodal vector field, the next view shows one volume-weighted magnetization vector per grain. Each arrow is centred on the volume-weighted grain centre, its direction is the mean magnetization direction, and its length represents the magnitude of the mean. Use the slider to select a TOMMOS snapshot; the corresponding applied field is shown above the plot. Snapshots are loaded on demand and cached only as four averaged vectors, so the complete hysteresis sequence does not have to remain in memory.

In [ ]:
# NBVAL_SKIP
def grain_average_data(state):
    """Return one volume-weighted magnetization vector per grain."""
    state_material_ids = np.asarray(state.cell_data["mat_id"])
    centres = []
    mean_magnetizations = []

    for material_id in magnetic_material_ids:
        grain = state.extract_cells(state_material_ids == material_id)
        grain_cells = grain.point_data_to_cell_data().compute_cell_sizes(
            length=False, area=False, volume=True
        )
        volumes = np.asarray(grain_cells.cell_data["Volume"])
        centres.append(
            np.average(grain.cell_centers().points, axis=0, weights=volumes)
        )
        mean_magnetizations.append(
            np.average(
                np.asarray(grain_cells.cell_data["m"]),
                axis=0,
                weights=volumes,
            )
        )

    mean_magnetizations = np.asarray(mean_magnetizations)
    averages = pv.PolyData(np.asarray(centres))
    averages.point_data["m_mean"] = mean_magnetizations
    averages.point_data["m_mean_norm"] = np.linalg.norm(
        mean_magnetizations, axis=1
    )
    averages.point_data["m_mean_z"] = mean_magnetizations[:, 2]
    averages.point_data["grain_id"] = magnetic_material_ids
    return averages


centred_arrow = pv.Arrow(start=(-0.5, 0, 0), direction=(1, 0, 0))


def average_glyphs(averages):
    """Create one arrow for every grain average."""
    return averages.glyph(
        orient="m_mean",
        scale="m_mean_norm",
        factor=arrow_length,
        geom=centred_arrow,
    )


slider_history = np.atleast_1d(
    np.genfromtxt(
        output_directory / "hysteresis.csv", delimiter=",", names=True
    )
)
slider_fields = np.asarray(slider_history["B_ext_T"])
slider_snapshots = [
    output_directory
    / f"state_cfg{int(config):05d}_B{field:+.4e}T.vtu"
    for config, field in zip(
        slider_history["config"], slider_fields, strict=True
    )
]
missing_snapshots = [path for path in slider_snapshots if not path.exists()]
if missing_snapshots:
    raise RuntimeError(f"Missing TOMMOS snapshot: {missing_snapshots[0]}")
if len(slider_snapshots) < 2:
    raise RuntimeError("At least two TOMMOS snapshots are required for the slider.")

domain_lengths = (
    magnetic_result_nm.bounds[1] - magnetic_result_nm.bounds[0],
    magnetic_result_nm.bounds[3] - magnetic_result_nm.bounds[2],
    magnetic_result_nm.bounds[5] - magnetic_result_nm.bounds[4],
)
arrow_length = 0.2 * max(domain_lengths)
initial_snapshot = len(slider_snapshots) - 1
initial_state = (
    result_nm
    if slider_snapshots[initial_snapshot] == snapshots[-1]
    else pv.read(slider_snapshots[initial_snapshot])
)
grain_average_cache = {initial_snapshot: grain_average_data(initial_state)}
mean_glyphs = average_glyphs(grain_average_cache[initial_snapshot])

grain_average_plotter = pv.Plotter()
grain_average_plotter.add_mesh(
    magnetic_result_nm.extract_surface(algorithm="dataset_surface"),
    scalars="mat_id",
    categories=True,
    cmap="tab10",
    opacity=0.2,
)
grain_average_plotter.add_mesh(
    mean_glyphs,
    scalars="m_mean_z",
    cmap="coolwarm",
    clim=(-1, 1),
    scalar_bar_args={"title": "mean m_z"},
)
field_label = grain_average_plotter.add_text(
    f"B = {slider_fields[initial_snapshot]:+.4f} T",
    position="upper_left",
)


def update_grain_averages(snapshot_index):
    """Update the arrows when the snapshot slider is released."""
    snapshot_index = int(
        np.clip(round(snapshot_index), 0, len(slider_snapshots) - 1)
    )
    if snapshot_index not in grain_average_cache:
        state = pv.read(slider_snapshots[snapshot_index])
        grain_average_cache[snapshot_index] = grain_average_data(state)

    mean_glyphs.copy_from(
        average_glyphs(grain_average_cache[snapshot_index]), deep=True
    )
    field_label.set_text(
        "upper_left",
        f"B = {slider_fields[snapshot_index]:+.4f} T",
    )
    grain_average_plotter.render()


grain_average_plotter.add_slider_widget(
    update_grain_averages,
    rng=(0, len(slider_snapshots) - 1),
    value=initial_snapshot,
    title="snapshot",
    fmt="%.0f",
    interaction_event="end",
)
grain_average_plotter.add_axes()
grain_average_plotter.show(jupyter_backend="trame")
# TODO: make as short as possible and use possibly a ipywidget slider

### Read the scalar hysteresis data

The magnetization fields are stored in VTU files, whereas the field sweep and volume-averaged quantities are stored in `hysteresis.csv`. Until the adapter populates `system.table`, the CSV file can be inspected directly.

In [ ]:
system.table.data.head()

### Plot the hysteresis branch

The TOMMOS columns `B_ext_T` and `J_par_T` are signed scalar projections along TOMMOS's field direction $\mathbf{h}$. The adapter defines $\mathbf{h}$ from `H_start` towards `H_stop`; in this example $\mathbf{h}=-\mathbf{e}_z$. Consequently, the raw CSV signs are opposite to the components measured along the laboratory $+z$ axis. The plot below projects both quantities back onto $+z$ using $\mathbf{h}\cdot\mathbf{e}_z=-1$. It is important to transform both axes, not only the magnetization.

The current calculation contains a single sweep from the initial to the final field and therefore represents one hysteresis branch rather than a closed loop. The chosen field discretization is intended as a technical test; a physically resolved curve requires more field points.

In [ ]:
# NBVAL_SKIP
system.table.mpl(y=["J_par"])

## MaMMoS use mammos-analysis tools to extract the remanence, coercive field and BHmax

In [ ]:
# TODO: maybe here or single grain notebook only??